# NumPy arrays: reliable and resource-aware computation

Fast, valid code can still produce a misleading result. **Core path: 45 minutes.**

## Outcomes

By the end, you can:

- recognize misleading results caused by representation or missing values;
- choose a meaningful numerical comparison and reproducible random stream;
- estimate an array computation's memory cost before running it;
- choose an import and storage path that fits the data and access pattern.

## Setup

Continue with the same temperature measurements: axes `(time, y, x)`, values in degrees Celsius.

In [1]:
import numpy as np

temperature = np.load("temperature_grid.npy", allow_pickle=False)

temperature.shape, temperature.dtype, temperature.nbytes

((2, 3, 4), dtype('float64'), 192)

## Integer range and conversion

An 8-bit sensor count is compact—but its range is finite.

In [2]:
raw_counts = np.array([240, 250], dtype=np.uint8)
limits = np.iinfo(raw_counts.dtype)

raw_counts, raw_counts.dtype, (limits.min, limits.max)

(array([240, 250], dtype=uint8), dtype('uint8'), (0, 255))

### Predict before running

A calibration adds 20 counts. Will the result be `[260, 270]`, an error, or something else?

In [3]:
raw_counts + np.uint8(20)

array([ 4, 14], dtype=uint8)

The operation is valid for the dtype, but wrong for the measurement. Widen before arithmetic.

In [4]:
wide_counts = raw_counts.astype(np.uint16)
corrected_counts = wide_counts + 20

corrected_counts, corrected_counts.dtype

(array([260, 270], dtype=uint16), dtype('uint16'))

Conversion can also discard information: floating-point values are truncated when converted to integers.

In [5]:
sample = temperature[0, 0]
sample, sample.astype(np.int16)

(array([18.2, 18.4, 18.7, 19. ]), array([18, 18, 18, 19], dtype=int16))

## Floating-point precision

- Lower precision saves memory and changes representable values. Whether that matters depends on the scientific scale.
- Extended types such as `longdouble` are platform-dependent.

In [6]:
temperature16 = temperature.astype(np.float16)
maximum_error = np.abs(temperature - temperature16).max()

{
    "float64 bytes": temperature.nbytes,
    "float16 bytes": temperature16.nbytes,
    "maximum absolute error (°C)": float(maximum_error),
}

{'float64 bytes': 192,
 'float16 bytes': 48,
 'maximum absolute error (°C)': 0.006250000000001421}

### Non-finite values

`NaN` can mark a missing or invalid observation; infinity can arise from overflow or division. Both are non-finite.

In [7]:
observed = temperature[0].copy()
observed[0, 1] = np.nan
observed[0, 2] = np.inf

{
    "ordinary mean": float(observed.mean()),
    "NaN values": int(np.isnan(observed).sum()),
    "finite values": int(np.isfinite(observed).sum()),
    "all finite": bool(np.isfinite(observed).all()),
    "NaN-aware mean": float(np.nanmean(observed)),
}

{'ordinary mean': nan,
 'NaN values': 1,
 'finite values': 10,
 'all finite': False,
 'NaN-aware mean': inf}

`nanmean` skips `NaN`, not infinity. Any omission is a scientific policy, not an automatic repair.

## Compare computed results

For computed value $a$ and reference value $b$, `isclose` accepts $|a-b| \leq a_{\mathrm{tol}} + r_{\mathrm{tol}} |b|$. Here $a_{\mathrm{tol}}$ is the absolute tolerance and $r_{\mathrm{tol}}$ the relative tolerance.

### Predict before running

One recomputed temperature differs by 0.03 °C. Which checks accept it?

In [8]:
reference = temperature[0].copy()
recomputed = reference.copy()
recomputed[1, 2] += 0.03

{
    "exactly equal": np.array_equal(recomputed, reference),
    "within 0.01 °C": np.allclose(recomputed, reference, rtol=0.0, atol=0.01),
    "within 0.05 °C": np.allclose(recomputed, reference, rtol=0.0, atol=0.05),
}

{'exactly equal': False, 'within 0.01 °C': False, 'within 0.05 °C': True}

A tolerance needs a scientific justification—instrument resolution, numerical error, or a stated acceptance criterion.

## Random does not mean unreproducible

A generator owns state. Equal seeds initialize equal streams.

In [9]:
seed = 20261105
rng_a = np.random.default_rng(seed)
rng_b = np.random.default_rng(seed)
noise_a = rng_a.normal(loc=0.0, scale=0.05, size=temperature[0].shape)
noise_b = rng_b.normal(loc=0.0, scale=0.05, size=temperature[0].shape)

{
    "bit generator": type(rng_a.bit_generator).__name__,
    "same first draw": np.array_equal(noise_a, noise_b),
    "first row (°C)": np.round(noise_a[0], 3),
}

{'bit generator': 'PCG64',
 'same first draw': True,
 'first row (°C)': array([-0.007, -0.024, -0.039,  0.032])}

A seed is necessary, not sufficient: also record the bit generator, environment, inputs, and analysis choices.

### Predict before running

Will a second draw from `rng_a` repeat its first draw?

In [10]:
noise_next = rng_a.normal(loc=0.0, scale=0.05, size=temperature[0].shape)

np.array_equal(noise_a, noise_next), np.round(noise_next[0], 3)

(False, array([0.008, 0.056, 0.086, 0.051]))

## Memory before speed

Vectorized does not mean free. For example, `calibrated = (measurements - offset) * scale`, where `offset` and `scale` are scalar calibration constants:

- input: `measurements`;
- temporary: `measurements - offset`;
- result: `calibrated`.

During the multiplication, all three arrays may coexist in memory.

In [11]:
large_shape = (365, 2_000, 2_000)
element_count = np.prod(large_shape, dtype=np.int64)
one_array_gib = element_count * np.dtype(np.float64).itemsize / 1024**3

{
    "shape": large_shape,
    "one float64 array (GiB)": float(round(one_array_gib, 1)),
    "input + temporary + result (GiB)": float(round(3 * one_array_gib, 1)),
}

{'shape': (365, 2000, 2000),
 'one float64 array (GiB)': 10.9,
 'input + temporary + result (GiB)': 32.6}

### Broadcasting can amplify

Inspect the implied result shape without allocating the result.

In [12]:
time_bias_shape = (365, 1, 1)
spatial_field_shape = (1, 2_000, 2_000)
result_shape = np.broadcast_shapes(time_bias_shape, spatial_field_shape)
itemsize = np.dtype(np.float64).itemsize
input_gib = (np.prod(time_bias_shape) + np.prod(spatial_field_shape)) * itemsize / 1024**3
result_gib = np.prod(result_shape) * itemsize / 1024**3

{
    "input arrays (GiB)": float(round(input_gib, 3)),
    "result shape": result_shape,
    "materialized result (GiB)": float(round(result_gib, 1)),
}

{'input arrays (GiB)': 0.03,
 'result shape': (365, 2000, 2000),
 'materialized result (GiB)': 10.9}

- Start with clear array code.
- Measure runtime and memory before optimizing.
- Use in-place operations or chunking only when measurements justify the added complexity.

## I/O is part of the computation

Text is often the interchange boundary. Repeated analysis needs a deliberate representation and access pattern.

### Import text deliberately

The CSV stores only the morning $(y, x)$ slice. Its comment records units, axes, and time—metadata that the numeric table itself does not preserve.

In [13]:
text_temperature = np.loadtxt("temperature_morning.csv", delimiter=",")

text_temperature.shape, text_temperature.dtype, text_temperature

((3, 4),
 dtype('float64'),
 array([[18.2, 18.4, 18.7, 19. ],
        [17.8, 18. , 18.2, 18.5],
        [17.3, 17.6, 17.8, 18. ]]))

Validate imported structure and values; successful parsing is not enough.

In [14]:
{
    "expected shape": temperature[0].shape,
    "shape matches": text_temperature.shape == temperature[0].shape,
    "values match within 0.05 °C": np.allclose(
        text_temperature, temperature[0], rtol=0.0, atol=0.05
    ),
}

{'expected shape': (3, 4),
 'shape matches': True,
 'values match within 0.05 °C': True}

### Choose for the data lifecycle

- `loadtxt`: regular numeric tables.
- `genfromtxt`: missing values or named fields, with more parsing and dtype decisions.

- Text/CSV (`savetxt`/`loadtxt`): inspectable and interoperable; parsing and schema reconstruction cost time.
- `.npy` (`save`/`load`): faithful shape and dtype for one NumPy array; little scientific metadata.
- `.npz`: several small arrays; no chunked partial access within an array.
- HDF5/NetCDF/Zarr: structured or chunked datasets; performance still depends on layout and access.

A read-only memory map can avoid eagerly loading a complete `.npy` array.

In [15]:
mapped_temperature = np.load(
    "temperature_grid.npy", mmap_mode="r", allow_pickle=False
)

(
    type(mapped_temperature).__name__,
    mapped_temperature.shape,
    mapped_temperature.dtype,
    mapped_temperature.flags.writeable,
)

('memmap', (2, 3, 4), dtype('float64'), False)

Memory mapping changes loading behavior; it does not turn `.npy` into a scalable parallel-I/O format.

### HPC: convert once, read deliberately

- Parse external text once; validate it, then convert for repeated analysis.
- Avoid every rank—every parallel process—parsing the same large text file.
- Avoid many tiny files; choose per-rank (one writer per process), aggregated (a few writers), or collective (coordinated parallel) I/O deliberately.
- Measure on the target filesystem: a format name does not guarantee performance.

## Decision check

Before trusting a result:

- Is the dtype's range and precision suitable?
- Are all required values finite?
- Is the tolerance tied to the scientific question?
- Can the random stream and environment be reconstructed?
- How large are the result and temporary arrays?
- Which schema and metadata survive the file boundary?
- Does the access pattern scale to the process count and filesystem?

## Takeaways

- A valid NumPy operation can still be scientifically wrong.
- Reliability requires explicit representation, comparison, and missing-data choices.
- Shape reasoning predicts memory as well as correctness.
- I/O choices determine fidelity, reproducibility, and scalability.